In [0]:
%sql
create catalog bus;
create database bus.data;


In [0]:
%sql
create volume bus.data.checkp1;

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import *

BOOTSTRAP="bore.pub:30092"
TOPIC="BUS"
CHECKPOINT="/Volumes/bus/data/checkp1"
TARGET="bus.data.bronze_bus_positions"
schema=StructType([
    StructField("vehicle_id",StringType()),
    StructField("trip_id",StringType()), 
    StructField("start_time",StringType()),
    StructField("start_date",StringType()),
    StructField("schedule_relationship",IntegerType()),
    StructField("route_id",StringType()),
    StructField("latitude",DoubleType()),
    StructField("longitute",DoubleType()),
    StructField("speed",DoubleType()),
    StructField("pulled_at",LongType()),
    StructField("ingest_at",LongType()),
    StructField("event_ts",LongType())
])
raw=(spark.readStream.format("kafka")\
    .option("kafka.bootstrap.servers",BOOTSTRAP)\
    .option("subscribe",TOPIC)\
    .option("startingOffsets","earliest")\
    .option("failOnDataLoss","false")\
     .load()
    )
bronze=(raw.select (F.col("key").cast("string").alias("kafka_key"),
    F.col("value").cast("string").alias("raw_json"),
    "partition",
     "offset",
     "topic",
    F.col("timestamp").alias("kafka_ts"))
.withColumn("j",F.from_json("raw_json",schema))
.select("kafka_key","raw_json","partition","offset","topic","j.*"))
q=(bronze.writeStream.format('delta')\
        .option("checkpointLocation",CHECKPOINT)\
        .trigger(availableNow=True)\
        .toTable(TARGET))
q.awaitTermination()